# 01 — Calibrated Excel → BioSensor Feature Studio

**Synthetic demonstration.** This notebook cannot reconstruct raw Cytation calibration or plate maps. Adjust `configs/...json` for each real workbook and inspect the data before using results in JBE.

### Setup: notebook works from project root

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'biosensor_pipeline').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0,str(ROOT))
print('Project directory:',ROOT)

### Inspect the Excel workbook without modifying it

In [ ]:
from biosensor_pipeline.excel_extract import inspect_excel
workbook = ROOT / 'examples/synthetic_MEFL_timecourses.xlsx'
for sheet in inspect_excel(workbook,n_preview=2):
    print(sheet['sheet'], sheet['preview_rows'])

### Extract the frozen-style observation windows

In [ ]:
from biosensor_pipeline.excel_extract import extract_workbook
import pandas as pd
csv,qc,provenance = extract_workbook(workbook,
    ROOT / 'configs/example_extraction.json', ROOT / 'outputs/notebook_demo')
features_input = pd.read_csv(csv)
qc_report = pd.read_csv(qc)
display(features_input.head(6), qc_report.head(6))
print('Feature Studio import CSV:',csv)
print('Provenance JSON:',provenance)

### Inspect per-dose replicate responses

In [ ]:
import matplotlib.pyplot as plt
fig,axs=plt.subplots(1,2,figsize=(11,4),layout='constrained')
for ax,(key,part) in zip(axs,features_input.groupby('system')):
    for rep,sub in part.groupby('replicate'):
        ax.scatter(sub.dose,sub.signal,label=rep,alpha=.65)
    ax.set_title(f'{key} — synthetic, not experimental')
    ax.set_xlabel('Inducer (uM)' if key=='FdeR' else 'Inducer (nM)')
    ax.set_ylabel('MEFL/cell'); ax.grid(alpha=.2); ax.legend()
plt.show()

### Next: use the existing Feature Studio scientific engine

In [ ]:
print('Open: https://sb2cl.github.io/biosensor-feature-studio/')
print('In 01 / DATA upload:',csv.resolve())
print('Scientific feature fitting/bootstrapping is intentionally not duplicated in this notebook.')